In [1]:
from dotenv import load_dotenv
from openai import OpenAI

openai_client = OpenAI()
load_dotenv()

True

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [8]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from [https://ollama.com/download](https://ollama.com/download) for your operating system:
   - **macOS**: download and install the `.pkg`
   - **Windows**: download and install the `.msi`
   - **Linux**: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local Ollama server is running, you can also run:
```bash
curl http://localhost:11434
```

If you want to use it from Python, install the client with:
```bash
pip install ollama
```


In [ ]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages
)

In [4]:
print(response.output_text)

Yes—usually you can, **if the course is still open for enrollment** and you meet any requirements.

What to check:
- **Enrollment deadline**: Is registration still open?
- **Prerequisites**: Do you need prior knowledge or approval?
- **Access method**: Is it self-paced, or do you need an invite/code?
- **Capacity**: Some courses have limited spots.

If you tell me:
1. the **course name**,  
2. the **platform/school**, and  
3. whether you’re a **student or external learner**,  

I can help you figure out the exact steps to join.


In [3]:
def search(query):
    boost_dict = {'question':3.0, 'section':0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    results = index.search(
            query=query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
    )

    return results

In [4]:
query = 'how do I run ollama locally?'

search(query)

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: Introduction to LLMs and RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npi

In [ ]:
## We define a search 'tool' for the llm to use. 

## Here we outline what the tool is, give it a name, describe what it does, define it's parameters

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
} 

In [35]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it'}
]

In [ ]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool] ## we pass the tools here as a list
)

In [29]:
call = response.output[0]

In [36]:
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enrollment open start mid-course FAQ"}', call_id='call_xctqfXGnqCxl6xBnL7B7tcFv', name='search', type='function_call', id='fc_0f2a5ea7a70da9ef006a2d28ee8f1481a3aa655a760d8a7913', namespace=None, status='completed')

In [37]:
import json
args = json.loads(call.arguments)

In [38]:
results = search(**args)

## Sequence of Steps 

1. Make a call to the LLM <-- First call
2. LLM invokes search with the params we provide
3. Store the results of the search
4. Pass the results to the LLM  <-- Second Call
5. LLM uses the results to generate an answer
6. LLM returns the answer


### IMPORTANT
LLM's are stateless so we have to send the entire history in step 4 above or else it won't know what has come before.

In [39]:
result_json = json.dumps(results, indent=2)

In [40]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json

}

In [41]:
# Append the original message from call
messages.append(call)

In [42]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enrollment open start mid-course FAQ"}', call_id='call_xctqfXGnqCxl6xBnL7B7tcFv', name='search', type='function_call', id='fc_0f2a5ea7a70da9ef006a2d28ee8f1481a3aa655a760d8a7913', namespace=None, status='completed')]

In [43]:
## Add the function_call_output so the model knows what we've previously asked
messages.append(function_call_output)

In [44]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late can I join enrollment open start mid-course FAQ"}', call_id='call_xctqfXGnqCxl6xBnL7B7tcFv', name='search', type='function_call', id='fc_0f2a5ea7a70da9ef006a2d28ee8f1481a3aa655a760d8a7913', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_xctqfXGnqCxl6xBnL7B7tcFv',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get

In [45]:
#Now pass everything to the LLM to generate an informed response
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool] ## we pass the tools here as a list
)

In [47]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, make sure to submit your project while submissions are still being accepted.


In [48]:
response.usage

ResponseUsage(input_tokens=690, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=32, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=722)

# The Agentic Loop

If what if we want the agent to keep making api calls until it's satisfied? 

To do that, we update the sequence by adding a loop. 

## Sequence of Steps 

1. Make a call to the LLM <-- First call
2. LLM invokes search with the params we provide
3. We invoke the search
4. Send the results back to the LLM (another call)
5. LLM processes teh results 
6. LLM decides to make another tool call
7. Send one more API request
8. Process and return the answer

## Developer Prompt
To make this loop, we use a developer prompt to explicitly tell the the agent to make multiple searches. 

In [98]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
    
    result_json = json.dumps(result, indent=2)

    json_file = {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json

    }

    return json_file

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

Agents need: 
- a role; we are providing that in the instructions.
- memory; the messages serve that purpose. 
- tools; search-tools are what we're currently using. 

In [100]:
#Now pass everything to the LLM to generate an informed response
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool] ## we pass the tools here as a list
)

In [105]:

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

iterations = 1

while True:
    print(f'iteration #{iterations}')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool] ## we pass the tools here as a list
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            last_answer = item.content[0].text
            print(item.content[0].text)
    
    iterations = iterations+1
    if has_function_calls == False:
        break

iteration #1
function_call: search {"query":"join course discovered course enroll late registration admission FAQ"}
function_call: search {"query":"course enrollment can I join late discover course FAQ"}
iteration #2
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still being accepted. If you’re just learning without aiming for a certificate, you can start anytime.

If you’d like, I can also help with questions about registration, certificates, or homework submission. Any other areas you want to explore?


In [106]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [108]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:

    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    iterations = 1

    # Agentic Tool Call Loop
    while True:
        print(f'iteration #{iterations}')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool] ## we pass the tools here as a list
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)
        
        iterations = iterations+1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [110]:
result = agent_loop(instructions=instructions,model='gpt-5.4-mini', question=question )

iteration #1
function_call: search {"query":"join course discovered can I join enrollment registration FAQ"}
iteration #2
function_call: search {"query":"course live cohort certificate project submission accepting submissions FAQ self-paced join still discovering course"}
iteration #3
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you only follow it later in self-paced mode, you can still learn from it, but you won’t get a certificate.

If you want, I can also help with:
- whether registration is required,
- how certificates work,
- or how to catch up if you joined late.


In [112]:
question = "What's Queen's Gambit"
result = agent_loop(instructions=instructions,model='gpt-5.4-mini', question=question )

iteration #1
function_call: search {"query":"Queen's Gambit chess opening course FAQ"}
iteration #2
function_call: search {"query":"Queen's Gambit chess meaning opening FAQ gambit queen opening"}
iteration #3
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**. It starts with the moves:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a “gambit” to try to gain control of the center, especially the **e4** square.

A common idea is:
- White gives up the c-pawn temporarily
- In return, White aims for stronger central control and easier development

There are two main versions:
- **Queen’s Gambit Accepted**: Black takes the pawn with `...dxc4`
- **Queen’s Gambit Declined**: Black does not take it and instead supports the center

If you want, I can also explain:
- the **ideas behind Queen’s Gambit**
- the difference between **Accepted** and **Declined**
- or show you a **simple beginner line** to play it.


## Preventing off Topic Responses

Update the instructions to limit the Agent's scope

In [114]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results, and then perform more searches. 

Try to expand your search by using new keywords
based on the results you get from the search.

The question has to be about the course or its logistics, off topic questions shouldn't be answered. If the search returns nothing, it's likely
an off-topic. If you can't answer the question using FAQ, don't do it yourself. Only use the facts from the FAQ database.



At the end, ask if there are other areas that the user wants to explore.
"""

question = "What is the Queen's Gambit?"

In [115]:
result = agent_loop(instructions=instructions,model='gpt-5.4-mini', question=question )

iteration #1
function_call: search {"query":"Queen's Gambit"}
iteration #2
function_call: search {"query":"gambit queen course FAQ"}
iteration #3
ASSISTANT:
I couldn’t find any course FAQ entry about “the Queen’s Gambit,” so I’m not able to answer that from the course materials.

If you meant something else course-related, tell me the context and I can look again. Are there other areas you want to explore?


In [ ]:
# start here: https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=17